## Feature processing for timeseries analysis

- This notebook is focused on analyzing the mean behavior of the library, over time

- It outputs a timeseries dataframe and a steady state dataframe

- Most steady state analysis is in another notebook, however

In [ ]:
import paulssonlab.deaton.trenchripper.trenchripper as tr

import numpy as np
import pandas as pd
import seaborn as sns
import scipy as sp
import scipy.stats
from scipy.stats import ttest_ind

import sklearn as skl
import dask.dataframe as dd
import dask.array as da
import dask
import warnings
import copy
import random
from sklearn.metrics.pairwise import euclidean_distances,manhattan_distances,cosine_distances
from matplotlib.lines import Line2D

from sklearn.metrics import silhouette_score
import statsmodels.stats.multitest
from sklearn.linear_model import LinearRegression
from sklearn.cluster import AffinityPropagation
from sklearn.manifold import TSNE
from sklearn.cluster import AgglomerativeClustering
from dask.distributed import wait

from matplotlib import pyplot as plt
import ast


import pylab
import scipy.cluster.hierarchy as sch

import matplotlib.gridspec as gridspec
import matplotlib as mpl

import holoviews as hv
hv.extension('bokeh')

dask_sample_seed = 42
random.seed(42)
np.random.seed(42)


dask_wd = "/home/lag36/scratch/lag36/dask"

warnings.filterwarnings(action='once',category=UserWarning)

In [ ]:
def get_sgrnadf_from_scoredf(scoredf, feature_labels, time_label="final cell timepoints list", trenchid_label="Multi-Experiment Phenotype Trenchid"):
    scoredf_groupby = scoredf.groupby("sgRNA")
    sgrnadf = scoredf_groupby.apply(lambda x: x[trenchid_label].tolist()).to_frame().rename(columns={0:trenchid_label})
    
    for feature_label in feature_labels:    
        sgrnadf[feature_label + ": score"] = scoredf_groupby.apply(lambda x: np.array([val for item in x[feature_label + ": score"].tolist() for val in item]))
        
    sgrnadf[time_label] = scoredf_groupby.apply(lambda x: np.array([val for item in x[time_label].tolist() for val in item]))
    sgrnadf["Gene"] = scoredf_groupby.apply(lambda x: x["Gene"].iloc[0])
    sgrnadf["TargetID"] = scoredf_groupby.apply(lambda x: x["TargetID"].iloc[0])
    sgrnadf["N Mismatch"] = scoredf_groupby.apply(lambda x: x["N Mismatch"].iloc[0])
    sgrnadf["N Observations"] = scoredf_groupby.apply(lambda x: len(x[trenchid_label].tolist()))
    sgrnadf["Category"] = scoredf_groupby.apply(lambda x: x["Category"].iloc[0])
        
    return sgrnadf

def normalize_timeseries(feature_vector_series, lmbda=0.5):
    timeseries_arr = np.swapaxes(np.array(feature_vector_series.tolist()),1,2)
    sigma = np.std(timeseries_arr,axis=1)
    if lmbda>0.:
        sigma_prime = ((sigma + 1)**lmbda - 1) / lmbda ##yeo-johnson
    elif lmbda==0.:
        sigma_prime = np.log(sigma + 1)
    else:
        raise ValueError("lmbda cannot be negative")
    normalizer = sigma/sigma_prime
    normalized_timeseries = timeseries_arr/normalizer[:,np.newaxis,:]
    return normalized_timeseries

def get_timepoint_values(df, label, min_time, max_time, time_label="time (s)", trenchid_label="Multi-Experiment Phenotype Trenchid", flatten_vals=True):
    if flatten_vals:
        masked_label_series = df.apply(lambda x: np.array(x[label])[(np.array(x[time_label])>=min_time)*(np.array(x[time_label])<=max_time)], axis=1, meta="object")
        concat_series = masked_label_series.map_partitions(lambda x: np.concatenate(x.tolist()), meta=("Concat",object)).persist()
        delayed_series = concat_series.to_delayed()
        concat_da = [da.from_delayed(item, shape=(len(item.compute()),), dtype=float) for item in delayed_series]
        flattened_vals = da.concatenate(concat_da, axis=0).compute()        
        return flattened_vals
    else:
        masked_label_series = df.groupby(trenchid_label).apply(lambda x: np.array(x[label])[(np.array(x[time_label])>=min_time)*(np.array(x[time_label])<=max_time)], meta="object").persist()
        return masked_label_series

def get_feature_stats(df, feature_label, min_time, max_time,time_label="time (s)",trenchid_label="Multi-Experiment Phenotype Trenchid"):
    feature_vals = get_timepoint_values(df, feature_label, min_time, max_time,time_label=time_label,trenchid_label=trenchid_label)
    feature_median = np.median(feature_vals)
    feature_iqr = sp.stats.iqr(feature_vals)
    return feature_median,feature_iqr

def compute_score(df, feature_label, feature_median, feature_iqr):
    scores = 1.35*((df[feature_label] - feature_median)/feature_iqr)
    return scores

def get_feature_scores(df, feature_label, init_time_range=(0,28800), time_label="time (s)", trenchid_label="Multi-Experiment Phenotype Trenchid"):
    feature_median,feature_iqr = get_feature_stats(df,feature_label,init_time_range[0],init_time_range[1],time_label=time_label,trenchid_label=trenchid_label)
    scores = compute_score(df, feature_label, feature_median, feature_iqr)
    return scores

def get_all_feature_scores(df, feature_labels, init_time_range=(0,28800), clipping_range=(-4,4), time_label="time (s)", trenchid_label="Multi-Experiment Phenotype Trenchid"):
    
    for feature_label in feature_labels:
        print(feature_label)
        feature_scores = get_feature_scores(df, feature_label, init_time_range=init_time_range, time_label=time_label, trenchid_label=trenchid_label)
        feature_scores = np.clip(feature_scores,clipping_range[0],clipping_range[1])
        df[feature_label + ": z score"] = feature_scores.persist()
    
    return df

def get_l_norm(x,right_tail_only=False):
    if right_tail_only:
        x_median = np.nanmedian(x)
        x_vals = x[x>x_median]
    else:
        x_vals = x
    l_norm = sp.stats.yeojohnson_normmax(x_vals)
    return l_norm

def apply_transform(final_output_df_filtered,yeo_subsample = 1,dask_sample_seed=dask_sample_seed,early_time_cutoff=28800,steady_state_time_cutoff=64800,clipping_range=(-4,4),\
                       params_to_transform = ['Division: major_axis_length'], transform_list=["YJ-Right"], time_label="time (s)",trenchid_label="Multi-Experiment Phenotype Trenchid"):
    
    subsample_df = final_output_df_filtered.sample(frac=yeo_subsample, random_state = dask_sample_seed).persist()
    subsample_df = subsample_df[subsample_df[time_label]>=steady_state_time_cutoff]
    for i,param in enumerate(params_to_transform):
        print(param)
        transform = transform_list[i]
        all_param_values = subsample_df[param].astype(float).compute().tolist()
        all_param_values = np.array(all_param_values)
        all_param_values = all_param_values[~np.isnan(all_param_values)]
        
        if transform == "YJ" or transform == "YJ-Right":
            if transform == "YJ":
                l_norm = get_l_norm(all_param_values,right_tail_only=False)
            elif transform == "YJ-Right":
                l_norm = get_l_norm(all_param_values,right_tail_only=True)
            final_output_df_filtered[param + ": Transformed"] = final_output_df_filtered[param].apply(lambda x: sp.stats.yeojohnson(x,lmbda = l_norm), meta=(param + ": Transformed",float)).persist()
        elif transform == "Quant":
            all_param_values = final_output_df_filtered[param].astype(float).compute()
            quant_transformer = skl.preprocessing.QuantileTransformer(n_quantiles=1000,output_distribution='normal',subsample=len(all_param_values))
            quant_transformed_vals = quant_transformer.fit_transform(all_param_values[:,np.newaxis]).flatten()
            quant_transformed_vals = pd.Series(data=quant_transformed_vals)
            quant_transformed_vals.index = final_output_df_filtered.index.compute()
            final_output_df_filtered[param + ": Transformed"] = quant_transformed_vals
            final_output_df_filtered[param + ": Transformed"] = final_output_df_filtered[param + ": Transformed"].persist()
        elif transform == "None":
            final_output_df_filtered[param + ": Transformed"] = final_output_df_filtered[param]
            
    final_output_df_filtered = get_all_feature_scores(final_output_df_filtered, [param + ": Transformed" for param in params_to_transform],\
                                                      init_time_range=(0, early_time_cutoff),clipping_range=clipping_range,\
                                                      time_label=time_label,trenchid_label=trenchid_label)
    final_output_df_filtered = final_output_df_filtered.reset_index().set_index(trenchid_label,drop=True)
    return final_output_df_filtered

### Initial Data Processing (Mean Timeseries)

- Here I use the following method to standardize data
    1) Raw data from the steady state period of the experiment are taken and Yeo-Johnson Transformed
    2) Then, values are standardized by the uninduced distribution
- Note for the Y-J transform, I only use the right skew of the distribution, to force attention to the right tail of the distribution
- Note that, for the final version of this analysis, I omit quantile transforms since they are difficult to explain and do not add much

### Thoughts

1) I don't like that the Y-J transform is acting on random raw values, this distribution is very noisey and does not represent the variability that will need to be standardized.
2) However, applying standardization to the aggregation across trenches also seems wrong, since outliers will be further upweighted.
3) A resonable middle ground appears (to me):
    - Produce a smoothed trace per trench (without any transformation/standardization)
    - Transform and standardize using the distributions of these trench-aggregated values
    - Then, aggregate these z-scores by variant
    - If this also gives bad results, I can try further aggregation (keep refering to the steady state ipynb where this worked well)


### Changes

2023-02-07 
- Set dask seed to 42 on imports test output on "Compute Yeo-Johnson Transform, Standardize Data and Save Checkpoint"
- Also on apply_transform

#### Start Dask

In [ ]:
dask_controller = tr.trcluster.dask_controller(
    walltime="04:00:00",
    local=False,
    n_workers=300,
    n_workers_min=100,
    memory="8GB",
    working_directory=dask_wd,
)
dask_controller.startdask()

# dask_controller = tr.trcluster.dask_controller(
#     walltime="04:00:00",
#     local=False,
#     n_workers=100,
#     n_workers_min=100,
#     memory="16GB",
#     working_directory=dask_wd,
# )
# dask_controller.startdask()

In [ ]:
dask_controller.displaydashboard()

### Kernel regression on each trench

In [ ]:
from statsmodels.nonparametric import kernel_regression
from scipy.stats import iqr
from statsmodels.nonparametric.smoothers_lowess import lowess
import sklearn as skl
from tslearn.clustering import TimeSeriesKMeans
from tslearn.preprocessing import TimeSeriesScalerMeanVariance

def timeseries_kernel_reg(df, y_label, min_tpt, max_tpt, kernel_bins, kernel_bandwidth, time_label="Final time (s)", trenchid_label="Multi-Experiment Phenotype Trenchid"):
    def kernel_reg(x_arr,y_arr,start=min_tpt,end=max_tpt,kernel_bins=kernel_bins,kernel_bandwidth=kernel_bandwidth):
        intervals = np.linspace(start, end, num=kernel_bins, dtype=float)
        w = kernel_regression.KernelReg(y_arr,x_arr,"c",reg_type="lc",bw=np.array([kernel_bandwidth]),ckertype="gaussian").fit(intervals)[0]
        reg_x, reg_y = (intervals, w)
        return reg_x, reg_y
    
    # if time_label == "Cell Cycle":
    #     # if this is the cell cycle df, do the following
    #     kernel_result = df.groupby(trenchid_label).apply(lambda x: kernel_reg((x["Final time (s)"].values - (x["Delta time (s)"].values/2)),x[y_label].values,)[1],meta=(y_label, object),)
    # else:
    kernel_result = df.groupby(trenchid_label).apply(lambda x: kernel_reg(x[time_label].values,x[y_label].values,)[1],meta=(y_label, object),)
    return kernel_result


def get_all_kernel_regs(df, y_label_list, min_tpt = 0, max_tpt = 36000, kernel_bins=20, kernel_bandwidth = 7200, time_label="Cell Cycle", trenchid_label="Multi-Experiment Phenotype Trenchid"):
    out_df = copy.copy(df)

    for y_label in y_label_list:
        kernel_result = timeseries_kernel_reg(out_df, y_label, min_tpt, max_tpt, kernel_bins, kernel_bandwidth, time_label=time_label, trenchid_label=trenchid_label)
        out_df["Kernel Trace: " + y_label] = kernel_result.persist()
        
    return out_df

In [ ]:
# cell_cycle_df = dd.read_parquet("/home/de64/scratch/de64/sync_folder/2023-01-18_lDE20_Merged_Analysis/2023-01-18_lDE20_Lineage_Cell_Cycle_Merged/", engine="pyarrow",calculate_divisions=True)
# timepoints_df = dd.read_parquet("/home/de64/scratch/de64/sync_folder/2023-01-18_lDE20_Merged_Analysis/2023-01-18_lDE20_Lineage_Observations_Merged/", engine="pyarrow",calculate_divisions=True)
# growth_df = dd.read_parquet("/home/de64/scratch/de64/sync_folder/2023-01-18_lDE20_Merged_Analysis/2023-01-18_lDE20_Lineage_Growth_Observations_Merged/", engine="pyarrow",calculate_divisions=True)

cell_cycle_df = dd.read_parquet("/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/2023-04-24_Lineage_Cell_Cycle_Merged/",
                                engine="pyarrow",calculate_divisions=True)
timepoints_df = dd.read_parquet("/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/2023-04-24_Lineage_Observations_Merged/",
                                engine="pyarrow",calculate_divisions=True)
growth_df = dd.read_parquet("/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/2023-04-24_Lineage_Growth_Observations_Merged/",
                            engine="pyarrow",calculate_divisions=True)

In [ ]:
timepoints_df_path = "/home/lag36/scratch/lag36/2023-03-11_lDE26_Run_1/2023-04-21_lDE26_Run_1_Lineage_Observations"
timepoints_df = dd.read_parquet(timepoints_df_path,
                                engine="pyarrow",calculate_divisions=True)


barcode_df_path = "/home/lag36/scratch/lag36/2023-03-11_lDE26_Run_1/2023-04-21_lDE26_Run_1_Final_Barcodes_df"
barcode_df = dd.read_parquet(barcode_df_path,
                                engine="pyarrow",calculate_divisions=True)


In [ ]:
df_pre = timepoints_df[timepoints_df['Observation time (s)'] <3600*2].compute()

In [ ]:
barcode_df_subset = barcode_df[["phenotype trenchid","Gene","oDEPool7_id"]].set_index("phenotype trenchid",sorted=True).groupby("phenotype trenchid").first().compute()

In [ ]:
barcode_df_subset

In [ ]:
test_df = pd.merge(df_pre,barcode_df_subset,on="phenotype trenchid",how="inner")

In [ ]:
plt.hist(test_df["major_axis_length"],range=(3,6),bins=20)

In [ ]:
plt.hist(test_df.groupby("phenotype trenchid")["major_axis_length"].mean(),range=(3.8,5.2),bins=30,log=True)

In [ ]:
plt.hist(test_df.groupby("oDEPool7_id")["major_axis_length"].mean(),range=(3.8,5.2),bins=30,log=True)

In [ ]:
post28_df = dd.read_parquet("/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/Growth_Division/2023-04-20_lDE28_Final_FOV_and_Time_Normalization",calculate_divisions=True)
early28_df = post28_df[post28_df["timepoints"]<12]
lDE28_after_2 = early28_df.groupby("trenchid")["major_axis_length"].mean().compute()

In [ ]:
post26_df = dd.read_parquet("/home/lag36/scratch/lag36/2023-03-11_lDE26_Run_1/Growth_Division/2023-04-18_lDE26_Final_FOV_and_Time_Normalization",calculate_divisions=True)
early26_df = post26_df[post26_df["timepoints"]<12]
lDE26_after_2 = early26_df.groupby("trenchid")["major_axis_length"].mean().compute()

In [ ]:
post26_df["timepoints"].head()

In [ ]:
lDE28_after_2.value_counts()

In [ ]:
lDE26_after_2.value_counts()

In [ ]:
post28_df_before_5 = dd.read_parquet("/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/2023-03-25_lDE28_Run_1_Lineage_Observations",calculate_divisions=True)
post28_df_before_5_early = post28_df_before_5[post28_df_before_5["Observation time (s)"]<(3600*1)]
lDE28_before_5 = post28_df_before_5_early.groupby("phenotype trenchid")["major_axis_length"].mean().compute()

In [ ]:
dask_controller.shutdown()

In [ ]:
plt.hist(lDE28_before_5)

In [ ]:
post26_df_before_5 = dd.read_parquet("/home/lag36/scratch/lag36/2023-03-11_lDE26_Run_1/2023-04-21_lDE26_Run_1_Lineage_Observations",calculate_divisions=True)
post26_df_before_5_early = post26_df_before_5[post26_df_before_5["Observation time (s)"]<(3600*2.1)]
lDE26_before_5 = post26_df_before_5_early.groupby("phenotype trenchid")["major_axis_length"].mean().compute()

In [ ]:
lDE26_before_5.value_counts()

In [ ]:
plt.hist(lDE26_before_5)

In [ ]:
kymo_26 = dd.read_parquet("/home/lag36/scratch/lag36/2023-03-11_lDE26_Run_1/Growth_Division/kymograph/metadata",\
                calculate_divisions=True)

kymo_28 = dd.read_parquet("/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/Growth_Division/kymograph/metadata",\
                calculate_divisions=True)

In [ ]:
kymo_26.get_partition(80).compute()

In [ ]:
kymo_28.get_partition(0).compute()

In [ ]:
timepoints_df_26 = dd.read_parquet("/home/lag36/scratch/lag36/2023-03-11_lDE26_Run_1/2023-04-24_Lineage_Observations_Merged/",
                                engine="pyarrow",calculate_divisions=True)

In [ ]:
timepoints_df_28 = dd.read_parquet("/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/2023-04-24_Lineage_Observations_Merged/",
                                engine="pyarrow",calculate_divisions=True)

In [ ]:
timepoints_df_26[

In [ ]:
(timepoints_df_26[timepoints_df_26["Observation time (s)"]<2*3600]
 .groupby("phenotype trenchid")["major_axis_length"]
 .mean()
 .compute()
 .value_counts()
)

In [ ]:
(timepoints_df_28[timepoints_df_28["Observation time (s)"]<2*3600].
 groupby("phenotype trenchid")["major_axis_length"]
 .mean()
 .compute()
 .value_counts()
)

In [ ]:
hist26 = (timepoints_df_26[timepoints_df_26["Observation time (s)"]<2*3600]
 .groupby("phenotype trenchid")["major_axis_length"]
 .mean()
 .compute())
hist28 = (timepoints_df_28[timepoints_df_28["Observation time (s)"]<2*3600]
 .groupby("phenotype trenchid")["major_axis_length"]
 .mean()
 .compute())

In [ ]:
plt.hist(hist26)

In [ ]:
hist28

In [ ]:
lineage_df_28 = dd.read_parquet("/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/Growth_Division/lineage/withNucleoidData",calculate_divisions=True)

In [ ]:
preinduction_mean_28 = lineage_df_28[lineage_df_28["timepoints"]<12].groupby("trenchid")["major_axis_length"].mean().compute()

In [ ]:
plt.hist(preinduction_mean_28,range=(3.8,5.2),bins=30,log=True)

In [ ]:
# cell_cycle_params = ['Birth: major_axis_length', 'Division: major_axis_length',
#    'Delta: major_axis_length', 'Birth: minor_axis_length',
#    'Division: minor_axis_length', 'Delta: minor_axis_length',
#    'Birth: Volume', 'Division: Volume', 'Delta: Volume',
#    'Final timepoints', 'Delta Timepoints', 'Final time (s)',
#    'Delta time (s)','Septum Displacement','Septum Displacement Length Normalized']
# timepoints_params = ['area','major_axis_length', 'minor_axis_length', 'Volume','mCherry mean_intensity']
# growth_params = ['Instantaneous Growth Rate: Volume', 'Instantaneous Growth Rate: major_axis_length']

cell_cycle_params =['Birth: major_axis_length',
                    'Division: major_axis_length',
                    'Delta: major_axis_length',
                    'Septum Displacement',
                    'Septum Displacement Length Normalized',
                    'Birth: minor_axis_length',
                    'Division: minor_axis_length',
                    'Delta: minor_axis_length',
                    'Birth: Volume',
                    'Division: Volume',
                    'Delta: Volume',
                    'Birth: Total Nucleoid Area',
                    'Division: Total Nucleoid Area',
                    'Birth: Total Nucleoid Length',
                    'Division: Total Nucleoid Length',
                    'Birth: Nucleoid Area Mean',
                    'Division: Nucleoid Area Mean',
                    'Birth: Nucleoid Mean Intensity Mean',
                    'Division: Nucleoid Mean Intensity Mean',
                    'Birth: Nucleoid Major Axis Length Mean',
                    'Division: Nucleoid Major Axis Length Mean',
                    'Birth: Nucleoid Minor Axis Length Mean',
                    'Division: Nucleoid Minor Axis Length Mean',
                    'Birth: Total Nucleoid Length per Cell Length',
                    'Division: Total Nucleoid Length per Cell Length',
                    'Delta: Total Nucleoid Length per Cell Length',
                    'Birth: Total Nucleoid Area per Cell Area',
                    'Division: Total Nucleoid Area per Cell Area',
                    'Delta: Total Nucleoid Area per Cell Area',
                    'Birth: Total Number of Nucleoids per Cell Length',
                    'Division: Total Number of Nucleoids per Cell Length',
                    'Delta: Total Number of Nucleoids per Cell Length',
                    'Birth: Number of Nucleoids',
                    'Division: Number of Nucleoids',
                    'Birth: Nucleoid Separation Mean',
                    'Division: Nucleoid Separation Mean',
                    'Birth: Nucleoid Separation CV',
                    'Division: Nucleoid Separation CV',
                    'Birth: Nucleoid Separation Min',
                    'Division: Nucleoid Separation Min',
                    'Birth: Nucleoid Separation Max',
                    'Division: Nucleoid Separation Max',
                    'Final timepoints',
                    'Delta Timepoints',
                    'Final time (s)',
                    'Delta time (s)']

timepoints_params =['area',
                    'major_axis_length',
                    'minor_axis_length',
                    'Volume',
                    'YFP mean_intensity',
                    'Total Nucleoid Area',
                    'Total Nucleoid Length',
                    'Nucleoid Area Mean',
                    'Nucleoid Mean Intensity Mean',
                    'Nucleoid Major Axis Length Mean',
                    'Nucleoid Minor Axis Length Mean',
                    'Total Nucleoid Length per Cell Length',
                    'Total Nucleoid Area per Cell Area',
                    'Total Number of Nucleoids per Cell Length',
                    'Number of Nucleoids',
                    'Nucleoid Separation Mean',
                    'Nucleoid Separation CV',
                    'Nucleoid Separation Min',
                    'Nucleoid Separation Max']

growth_params = ['Instantaneous Growth Rate: Volume',
                 'Instantaneous Growth Rate: major_axis_length']

In [ ]:
all_traced_params = cell_cycle_params + timepoints_params + growth_params
all_traced_params = ["Kernel Trace: " + item for item in all_traced_params]

bandwidth_hours = 2
kernel_bins = 20

cell_cycle_df = get_all_kernel_regs(
    cell_cycle_df,
    cell_cycle_params,
    min_tpt = 0,
    max_tpt = 36000,
    kernel_bins = kernel_bins,
    kernel_bandwidth = bandwidth_hours*3600,
    time_label="Mid-Cycle time (s)",
    trenchid_label="Multi-Experiment Phenotype Trenchid")

timepoints_df = get_all_kernel_regs(
    timepoints_df,
    timepoints_params,
    min_tpt = 0,
    max_tpt = 36000,
    kernel_bins = kernel_bins,
    kernel_bandwidth = bandwidth_hours*3600,
    time_label="Observation time (s)",
    trenchid_label="Multi-Experiment Phenotype Trenchid")

growth_df = get_all_kernel_regs(
    growth_df,
    growth_params,
    min_tpt = 0,
    max_tpt = 36000,
    kernel_bins = kernel_bins,
    kernel_bandwidth = bandwidth_hours*3600,
    time_label="Measurement time (s)",
    trenchid_label="Multi-Experiment Phenotype Trenchid")

In [ ]:
print(1)

#### Merge Timeseries dfs

In [ ]:
trenchid_label="Multi-Experiment Phenotype Trenchid"

cell_cycle_out = cell_cycle_df.groupby(trenchid_label).apply(lambda x: x.iloc[0])
timepoints_out = timepoints_df.groupby(trenchid_label).apply(lambda x: x.iloc[0])
growth_out = growth_df.groupby(trenchid_label).apply(lambda x: x.iloc[0])

traced_col_names = [col_name for col_name in cell_cycle_out.columns.tolist() if col_name in all_traced_params]
cell_cycle_out = cell_cycle_out[['Global CellID', 'File Parquet Index', 'fov', 'row', 'trench',
       'initial timepoints', 'File Index', 'File Trench Index', 'CellID',
       'Trench Score', 'Mother CellID', 'Daughter CellID 1',
       'Daughter CellID 2', 'Sister CellID', 'Centroid X', 'Centroid Y',
       'Kymograph File Parquet Index', 'Kymograph FOV Parquet Index',
       'FOV Parquet Index', 'Experiment #', 'phenotype trenchid'] + traced_col_names]

traced_col_names = [col_name for col_name in timepoints_out.columns.tolist() if col_name in all_traced_params]
timepoints_out = timepoints_out[traced_col_names]

traced_col_names = [col_name for col_name in growth_out.columns.tolist() if col_name in all_traced_params]
growth_out = growth_out[traced_col_names]

timeseries_merged_dd = dd.concat([cell_cycle_out,timepoints_out,growth_out], axis=1)
timeseries_merged_dd = timeseries_merged_dd.persist()
wait(timeseries_merged_dd);

#### Export Kernel Regression Dataframe

In [ ]:
kernel_trace_columns = [column_name for column_name in timeseries_merged_dd.columns if "Kernel Trace" in column_name]
explode_series_list = []
for column_name in kernel_trace_columns:
    working_exploded_column = timeseries_merged_dd[column_name].explode().astype(float).persist()
    explode_series_list.append(working_exploded_column)
kernel_output_df = dd.concat(explode_series_list, axis=1).persist()
wait(kernel_output_df);

n_indices = len(timeseries_merged_dd)
first_kernel_trace_column = [column_name for column_name in timeseries_merged_dd.columns if "Kernel Trace" in column_name][0]
timeseries_len = len(timeseries_merged_dd[first_kernel_trace_column].get_partition(0).compute().iloc[0])
timepoint_index = np.tile(range(timeseries_len),n_indices)
kernel_output_df = tr.add_list_to_column(kernel_output_df,list(timepoint_index),"Kernel Timepoints").persist()
wait(kernel_output_df);

kernel_output_df = kernel_output_df.reset_index()
kernel_output_df["Multi-Experiment Phenotype Trenchid-Kernel Timepoint Index"] =  kernel_output_df.apply(lambda x: int(f'{x["Multi-Experiment Phenotype Trenchid"].astype(int):010n}{x["Kernel Timepoints"]:03n}'), axis=1, meta=(None,'int64')).persist()
kernel_output_df = kernel_output_df.set_index("Multi-Experiment Phenotype Trenchid-Kernel Timepoint Index",sorted=True)
wait(kernel_output_df);

kernel_output_df.to_parquet("/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/2023-04-24_lDE28_Kernel_Regression_df",\
                            engine="pyarrow",schema="infer",overwrite=True)
dask_controller.daskclient.cancel(kernel_output_df)

In [ ]:
print(1)

### Compute Yeo-Johnson Transform, Standardize Data and Save Checkpoint

 - Still working on standardization
 - Switched IQR for std

In [ ]:
# cell_cycle_params_to_transform=['Delta time (s)','Septum Displacement Length Normalized']
# timepoints_params_to_transform=['major_axis_length','minor_axis_length','mCherry mean_intensity']
# growth_params_to_transform=['Instantaneous Growth Rate: Volume']
# params_to_transform_kernel_reg = ["Kernel Trace: " + item for item in (cell_cycle_params_to_transform+timepoints_params_to_transform+growth_params_to_transform)]
# transform_list = ["YJ","YJ","None","None","YJ","YJ"]


cell_cycle_params_to_transform=['Septum Displacement Length Normalized',
                                'Delta time (s)']

cell_cycle_params_to_transform_transfs = ["YJ",
                                          "YJ"]#"YJ-Right"]

timepoints_params_to_transform=['area',
                                'major_axis_length',
                                'minor_axis_length',
                                'YFP mean_intensity',
                                'Total Nucleoid Area',
                                'Total Nucleoid Length',
                                'Nucleoid Area Mean',
                                'Nucleoid Mean Intensity Mean',
                                'Nucleoid Major Axis Length Mean',
                                'Nucleoid Minor Axis Length Mean',
                                'Total Nucleoid Length per Cell Length',
                                'Total Nucleoid Area per Cell Area',
                                'Total Number of Nucleoids per Cell Length',
                                'Number of Nucleoids']

timepoints_params_to_transform_transfs = ["None",
                                            "None",
                                            "None",
                                            "YJ",
                                            "None",
                                            "None",
                                            "None",
                                            "None",#"YJ", # YJ Messes this one up!!
                                            "None",
                                            "None",
                                            "None",
                                            "None",
                                            "None", # TODO
                                            "None"] # TODO

# timepoints_params_to_transform_transfs = ["Quant",
#                                             "Quant",
#                                             "YJ",
#                                             "YJ-Right",
#                                             "Quant",
#                                             "Quant",
#                                             "Quant",
#                                             "YJ-Right",
#                                             "Quant",
#                                             "Quant",
#                                             "YJ",
#                                             "YJ",
#                                             "YJ",
#                                             "Quant"]

growth_params_to_transform=['Instantaneous Growth Rate: Volume']
growth_params_to_transform_transfs = ["YJ"]#"Quant"]

params_to_transform_kernel_reg = ["Kernel Trace: " + item for item in (cell_cycle_params_to_transform+timepoints_params_to_transform+growth_params_to_transform)]

transform_list = cell_cycle_params_to_transform_transfs + timepoints_params_to_transform_transfs + growth_params_to_transform_transfs


In [ ]:
min_tpt = 0
max_tpt = 36000
induction_time = 2*3600
steady_state_time = 8*3600
timepoint_step = (max_tpt-min_tpt)/kernel_bins
early_timepoint_cutoff = int(induction_time//timepoint_step)
steady_state_timepoint_cutoff = int(steady_state_time//timepoint_step)

kernel_output_df = dd.read_parquet("/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/2023-04-24_lDE28_Kernel_Regression_df/",engine="pyarrow",calculate_divisions=True)
kernel_output_transform_df = apply_transform(kernel_output_df,yeo_subsample = 1,early_time_cutoff=early_timepoint_cutoff,steady_state_time_cutoff=steady_state_timepoint_cutoff,\
                    clipping_range=(-6,6),params_to_transform = params_to_transform_kernel_reg, transform_list = transform_list,\
                                         time_label="Kernel Timepoints",trenchid_label="Multi-Experiment Phenotype Trenchid")
kernel_output_transform_df = kernel_output_transform_df.persist()
wait(kernel_output_transform_df);
kernel_output_transform_df.to_parquet("/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/2023-04-24_lDE28_Kernel_Regression_Transform_df", engine="pyarrow", overwrite=True)
dask_controller.daskclient.cancel(kernel_output_df)
dask_controller.daskclient.cancel(kernel_output_transform_df)
dask_controller.reset_worker_memory()

### Plot Transformed Features

In [ ]:
# cell_cycle_params_to_transform=['Delta time (s)','Septum Displacement Length Normalized']
# timepoints_params_to_transform=['major_axis_length','minor_axis_length','mCherry mean_intensity']
# growth_params_to_transform=['Instantaneous Growth Rate: Volume']
cell_cycle_params_to_transform=['Septum Displacement Length Normalized',
                                'Delta time (s)']

timepoints_params_to_transform=['area',
                                'major_axis_length',
                                'minor_axis_length',
                                'YFP mean_intensity',
                                'Total Nucleoid Area',
                                'Total Nucleoid Length',
                                'Nucleoid Area Mean',
                                'Nucleoid Mean Intensity Mean',
                                'Nucleoid Major Axis Length Mean',
                                'Nucleoid Minor Axis Length Mean',
                                'Total Nucleoid Length per Cell Length',
                                'Total Nucleoid Area per Cell Area',
                                'Total Number of Nucleoids per Cell Length',
                                'Number of Nucleoids']

growth_params_to_transform=['Instantaneous Growth Rate: Volume']

In [ ]:

params_to_transform_kernel_reg = ["Kernel Trace: " + item for item in (cell_cycle_params_to_transform+timepoints_params_to_transform+growth_params_to_transform)]

cell_cycle_df = dd.read_parquet("/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/2023-04-24_Lineage_Cell_Cycle_Merged", engine="pyarrow",calculate_divisions=True)
kernel_output_yj_df = dd.read_parquet("/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/2023-04-24_lDE28_Kernel_Regression_Transform_df", engine="pyarrow",calculate_divisions=True)

In [ ]:
sample_frac = 0.2 #0.01
kernel_output_yj_df_sampled = kernel_output_yj_df.sample(frac=sample_frac, random_state = dask_sample_seed).compute()

In [ ]:
ncols=3
nrows=12

# param_names = ["Interdivision Time (hr)","Septum Displacement",\
#               r"Length ($ \mu m $)",r"Width ($ \mu m $)",\
#               "mKate2 Intensity (AU)","Growth Rate (1/hr)"]
# param_ranges = [(0,5000),(0,0.05),(0.,10.),\
#                (1.1,1.7),(0,5000),(0,2)]
param_names = cell_cycle_params_to_transform + timepoints_params_to_transform + growth_params_to_transform
param_ranges = [(0,0.05),
                (0,5000),
                (0,15),
                (0,10),
                (0.5,1.5),
                (0,50000),
                (0,6),
                (0,7),
                (0,6),
                (0,15000),
                (0,6),
                (0.6,1.25),
                (0.4,1),
                (0,1),
                (0,1),
                (0,5),
                (0,2)]

fig = plt.figure(figsize=(8,18))
for i,param in enumerate(params_to_transform_kernel_reg):
    working_row = i//ncols
    param_name = param_names[i]
    param_range = param_ranges[i]
    param_transformed = param + ": Transformed: z score"
    vals = kernel_output_yj_df_sampled[param].values
    transformed_vals = kernel_output_yj_df_sampled[param_transformed].values
    
    plt.subplot(nrows,ncols,i+1)
    if i == 1:
        plt.title("Raw Values",fontsize=16)
    plt.hist(vals,bins=50,range=param_range,log=True)
    plt.xlabel(param_name)
    plt.subplot(nrows,ncols,i+1+(6*ncols))
    if i == 1:
        plt.title("Z-Scores",fontsize=16)
    plt.hist(transformed_vals,bins=50,range=(-10,10),log=True)
    plt.xlabel(param_name)
plt.tight_layout()
plt.savefig("/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/2023-04-24_lDE28_Timeseries_Zscore_Transformation.png",dpi=300)

In [ ]:
ncols=3
nrows=12

# param_names = ["Interdivision Time (hr)","Septum Displacement",\
#               r"Length ($ \mu m $)",r"Width ($ \mu m $)",\
#               "mKate2 Intensity (AU)","Growth Rate (1/hr)"]
# param_ranges = [(0,5000),(0,0.05),(0.,10.),\
#                (1.1,1.7),(0,5000),(0,2)]
param_names = cell_cycle_params_to_transform + timepoints_params_to_transform + growth_params_to_transform
param_ranges = [(0,0.05),
                (0,5000),
                (0,15),
                (0,10),
                (0.5,1.5),
                (0,50000),
                (0,6),
                (0,7),
                (0,6),
                (0,15000),
                (0,6),
                (0.6,1.25),
                (0.4,1),
                (0,1),
                (0,1),
                (0,5),
                (0,2)]

fig = plt.figure(figsize=(8,18))
for i,param in enumerate(params_to_transform_kernel_reg):
    working_row = i//ncols
    param_name = param_names[i]
    param_range = param_ranges[i]
    param_transformed = param + ": Transformed: z score"
    vals = kernel_output_yj_df_sampled[param].values
    transformed_vals = kernel_output_yj_df_sampled[param_transformed].values
    
    plt.subplot(nrows,ncols,i+1)
    if i == 1:
        plt.title("Raw Values",fontsize=16)
    plt.hist(vals,bins=50,range=param_range,log=True)
    plt.xlabel(param_name)
    plt.subplot(nrows,ncols,i+1+(6*ncols))
    if i == 1:
        plt.title("Z-Scores",fontsize=16)
    plt.hist(transformed_vals,bins=50,range=(-10,10),log=True)
    plt.xlabel(param_name)
plt.tight_layout()
plt.savefig("/home/lag36/scratch/lag36/2023-03-11_lDE26_Run_1/2023-04-24_Timeseries_Zscore_Transformation.png",dpi=300)

In [ ]:
ncols=3
nrows=12

# param_names = ["Interdivision Time (hr)","Septum Displacement",\
#               r"Length ($ \mu m $)",r"Width ($ \mu m $)",\
#               "mKate2 Intensity (AU)","Growth Rate (1/hr)"]
# param_ranges = [(0,5000),(0,0.05),(0.,10.),\
#                (1.1,1.7),(0,5000),(0,2)]
param_names = cell_cycle_params_to_transform + timepoints_params_to_transform + growth_params_to_transform
param_ranges = [(0,0.05),
                (0,5000),
                (0,15),
                (0,10),
                (0.5,1.5),
                (0,50000),
                (0,6),
                (0,7),
                (0,6),
                (0,15000),
                (0,6),
                (0.6,1.25),
                (0.4,1),
                (0,1),
                (0,1),
                (0,5),
                (0,2)]

fig = plt.figure(figsize=(8,18))
for i,param in enumerate(params_to_transform_kernel_reg):
    working_row = i//ncols
    param_name = param_names[i]
    param_range = param_ranges[i]
    param_transformed = param + ": Transformed: z score"
    vals = kernel_output_yj_df_sampled[param].values
    transformed_vals = kernel_output_yj_df_sampled[param_transformed].values
    
    plt.subplot(nrows,ncols,i+1)
    if i == 1:
        plt.title("Raw Values",fontsize=16)
    plt.hist(vals,bins=50,range=param_range,log=True)
    plt.xlabel(param_name)
    plt.subplot(nrows,ncols,i+1+(6*ncols))
    if i == 1:
        plt.title("Z-Scores",fontsize=16)
    plt.hist(transformed_vals,bins=50,range=(-6,6),log=True)
    plt.xlabel(param_name)
plt.tight_layout()
plt.savefig("/home/lag36/scratch/lag36/2023-03-11_lDE26_Run_1/2023-04-24_Timeseries_Zscore_Transformation.png",dpi=300)

#### Merge Timeseries dfs

In [ ]:
trenchid_label="Multi-Experiment Phenotype Trenchid"
kernel_score_params = [param + ": Transformed: z score" for param in params_to_transform_kernel_reg]
all_kernel_reg_columns = all_traced_params + kernel_score_params

kernel_output_yj_df_unexploded = kernel_output_yj_df.groupby(trenchid_label).apply(lambda x: pd.DataFrame({column: [np.array(x[column].tolist())] for column in x.columns}))
kernel_output_yj_df_unexploded = kernel_output_yj_df_unexploded[all_kernel_reg_columns]
kernel_output_yj_df_unexploded = kernel_output_yj_df_unexploded.persist()
wait(kernel_output_yj_df_unexploded);

cell_cycle_out = cell_cycle_df.groupby(trenchid_label).apply(lambda x: x.iloc[0])
cell_cycle_out = cell_cycle_out[['Global CellID', 'File Parquet Index', 'fov', 'row', 'trench',
       'initial timepoints', 'File Index', 'File Trench Index', 'CellID',
       'Trench Score', 'Mother CellID', 'Daughter CellID 1',
       'Daughter CellID 2', 'Sister CellID', 'Centroid X', 'Centroid Y',
       'Kymograph File Parquet Index', 'Kymograph FOV Parquet Index',
       'FOV Parquet Index', 'Experiment #', 'phenotype trenchid']]
cell_cycle_out = cell_cycle_out.persist()
wait(cell_cycle_out);

kernel_output_yj_df_unexploded = kernel_output_yj_df_unexploded.reset_index().set_index("Multi-Experiment Phenotype Trenchid", sorted=True)
kernel_output_yj_df_unexploded = kernel_output_yj_df_unexploded.drop("level_1",axis=1).persist()
wait(kernel_output_yj_df_unexploded);

timeseries_merged_dd = cell_cycle_out.join(kernel_output_yj_df_unexploded,on="Multi-Experiment Phenotype Trenchid")

#### Average over trenches for sgRNA-wise Timeseries

In [ ]:
final_output_df_barcodes = dd.read_parquet("/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/lDE28_Final_Barcodes_df_Merged/",engine="pyarrow",calculate_divisions=True)
final_output_df_barcodes = final_output_df_barcodes.groupby(trenchid_label,sort=False).apply(lambda x: x.iloc[0]).persist()
wait(final_output_df_barcodes);

final_output_df_barcodes = final_output_df_barcodes.reset_index().set_index("oDEPool7_id",drop=True)
final_output_df_barcodes = final_output_df_barcodes.reset_index().set_index(trenchid_label).persist()
wait(final_output_df_barcodes);
timeseries_merged_dd = timeseries_merged_dd.merge(final_output_df_barcodes[['oDEPool7_id','Barcode','sgRNA','Closest Hamming Distance', 'EcoWG1_id', 'Gene', 'N Mismatch', 'Category', 'TargetID', 'barcodeid']],how='inner',left_index=True,right_index=True).persist()

n_obs = timeseries_merged_dd.groupby("oDEPool7_id",sort=False).apply(lambda x: len(x.index.unique()), meta=int).compute()
n_obs = pd.DataFrame(n_obs).rename({0:"N Observations"}, axis=1).sort_index()
timeseries_merged_dd = timeseries_merged_dd.merge(n_obs,on="oDEPool7_id",how='inner',right_index=True)

feature_vector_series = timeseries_merged_dd.apply(lambda x: np.array(x[kernel_score_params].tolist()), axis=1)
timeseries_merged_dd["Feature Vector"] = feature_vector_series

timeseries_merged_dd_nan_filtered = timeseries_merged_dd[~timeseries_merged_dd["Feature Vector"].apply(lambda x: np.any(np.isnan(x)),meta=("No NaN",bool))]

#### Aggregate over timeseries

In [ ]:
## params
vectors_to_aggregate = ['Feature Vector'] + all_kernel_reg_columns

timeseries_merged_dd_nan_filtered_sgRNA_sorted = timeseries_merged_dd_nan_filtered.set_index("oDEPool7_id").persist()
timeseries_merged_dd_nan_filtered_sgRNA_sorted_groupby = timeseries_merged_dd_nan_filtered_sgRNA_sorted.groupby(["oDEPool7_id"])
    
sgRNA_df_merged = timeseries_merged_dd_nan_filtered_sgRNA_sorted_groupby.apply(lambda x: x.iloc[0]).persist()
sgRNA_df_merged = sgRNA_df_merged.compute()

for vector in vectors_to_aggregate:
    print(vector)
    mean_vectors = timeseries_merged_dd_nan_filtered_sgRNA_sorted_groupby.apply(lambda x: np.mean(x[vector], axis=0), meta=(None,object)).compute()
    sgRNA_df_merged[vector] = mean_vectors
    sem_vectors = timeseries_merged_dd_nan_filtered_sgRNA_sorted_groupby.apply(lambda x: sp.stats.sem(np.array(list(x[vector])), axis=0, nan_policy='propagate'), meta=(None,object)).compute()
    sgRNA_df_merged["SEM: " + vector] = sem_vectors
sgRNA_df_merged.to_pickle("/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/2023-04-24_lDE28_sgRNA_Timeseries_df.pkl")

In [ ]:
dask_controller.daskclient.cancel(cell_cycle_df)
dask_controller.daskclient.cancel(kernel_output_yj_df)
dask_controller.daskclient.cancel(timeseries_merged_dd)
dask_controller.daskclient.cancel(final_output_df_barcodes)
dask_controller.reset_worker_memory()

#### Get Steady State Measurements

In [ ]:
cell_cycle_df = dd.read_parquet("/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/2023-04-24_Lineage_Cell_Cycle_Merged", engine="pyarrow",calculate_divisions=True)
timepoints_df = dd.read_parquet("/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/2023-04-24_Lineage_Observations_Merged/", engine="pyarrow",calculate_divisions=True)
growth_df = dd.read_parquet("/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/2023-04-24_Lineage_Growth_Observations_Merged/", engine="pyarrow",calculate_divisions=True)


cell_cycle_params = ['Septum Displacement Length Normalized',
'Birth: minor_axis_length',
'Division: minor_axis_length',
'Delta: minor_axis_length',
'Birth: Volume',
'Division: Volume',
'Delta: Volume',
'Birth: Total Nucleoid Area',
'Division: Total Nucleoid Area',
'Delta: Total Nucleoid Area',
'Birth: Total Nucleoid Length',
'Division: Total Nucleoid Length',
'Delta: Total Nucleoid Length',
'Birth: Nucleoid Area Mean',
'Division: Nucleoid Area Mean',
'Birth: Nucleoid Mean Intensity Mean',
'Division: Nucleoid Mean Intensity Mean',
'Birth: Nucleoid Major Axis Length Mean',
'Division: Nucleoid Major Axis Length Mean',
'Birth: Nucleoid Minor Axis Length Mean',
'Division: Nucleoid Minor Axis Length Mean',
'Birth: Total Nucleoid Length per Cell Length',
'Division: Total Nucleoid Length per Cell Length',
'Birth: Total Nucleoid Area per Cell Area',
'Division: Total Nucleoid Area per Cell Area',
'Birth: Total Number of Nucleoids per Cell Length',
'Division: Total Number of Nucleoids per Cell Length',
'Birth: Number of Nucleoids',
'Division: Number of Nucleoids',
'Delta: Number of Nucleoids',
'Birth: Nucleoid Separation Mean',
'Division: Nucleoid Separation Mean',
'Birth: Nucleoid Separation CV',
'Division: Nucleoid Separation CV',
'Birth: Nucleoid Separation Min',
'Division: Nucleoid Separation Min',
'Birth: Nucleoid Separation Max',
'Division: Nucleoid Separation Max',
'Final timepoints',
'Delta Timepoints',
'Final time (s)',
'Delta time (s)']

timepoints_params = ['area',
'major_axis_length',
'minor_axis_length',
'Volume',
'YFP mean_intensity',
'Total Nucleoid Area',
'Total Nucleoid Length',
'Nucleoid Area Mean',
'Nucleoid Mean Intensity Mean',
'Nucleoid Major Axis Length Mean',
'Nucleoid Minor Axis Length Mean',
'Total Nucleoid Length per Cell Length',
'Total Nucleoid Area per Cell Area',
'Total Number of Nucleoids per Cell Length',
'Number of Nucleoids',
'Nucleoid Separation Mean',
'Nucleoid Separation CV',
'Nucleoid Separation Min',
'Nucleoid Separation Max']

growth_params = ['Instantaneous Growth Rate: Volume', 'Instantaneous Growth Rate: major_axis_length']

# cell_cycle_params = ['Birth: major_axis_length', 'Division: major_axis_length',
#    'Delta: major_axis_length', 'Birth: minor_axis_length',
#    'Division: minor_axis_length', 'Delta: minor_axis_length',
#    'Birth: Volume', 'Division: Volume', 'Delta: Volume',
#    'Final timepoints', 'Delta Timepoints', 'Final time (s)',
#    'Delta time (s)','Septum Displacement','Septum Displacement Length Normalized']
# timepoints_params = ['area','major_axis_length', 'minor_axis_length', 'Volume','mCherry mean_intensity']
# growth_params = ['Instantaneous Growth Rate: Volume', 'Instantaneous Growth Rate: major_axis_length']
all_traced_params = cell_cycle_params + timepoints_params + growth_params

#### Use Steady State Time to Filter

In [ ]:
# 2023-03-25_lDE28_Run_1/2023-04-24_lDE28

In [ ]:
steady_state_time = 8*3600 #8 hours
induction_time = 2*3600 #2 hours

trenchid_label="Multi-Experiment Phenotype Trenchid"

cell_cycle_df_ss = cell_cycle_df[cell_cycle_df["Initial time (s)"]>=steady_state_time]
cell_cycle_df_pre = cell_cycle_df[cell_cycle_df["Final time (s)"]<=induction_time]

timepoints_df_ss = timepoints_df[timepoints_df["Observation time (s)"]>=steady_state_time]
timepoints_df_pre = timepoints_df[timepoints_df["Observation time (s)"]<=induction_time]

growth_df_ss = growth_df[growth_df["Measurement time (s)"]>=steady_state_time]
growth_df_pre = growth_df[growth_df["Measurement time (s)"]<=induction_time]

#### Add sgRNA info for real data and save as checkpoint

In [ ]:
final_output_df_barcodes = dd.read_parquet("/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/lDE28_Final_Barcodes_df_Merged/",engine="pyarrow",calculate_divisions=True)
final_output_df_barcodes = final_output_df_barcodes.groupby(trenchid_label,sort=False).apply(lambda x: x.iloc[0]).persist()
wait(final_output_df_barcodes);

final_output_df_barcodes = final_output_df_barcodes.reset_index().set_index("oDEPool7_id",drop=True)
final_output_df_barcodes = final_output_df_barcodes.reset_index().set_index(trenchid_label).persist()
wait(final_output_df_barcodes);

cell_cycle_df_ss = cell_cycle_df_ss.join(final_output_df_barcodes[['oDEPool7_id','Barcode','sgRNA','Closest Hamming Distance', 'EcoWG1_id', 'Gene', 'N Mismatch', 'Category', 'TargetID', 'barcodeid']],how='inner')
cell_cycle_df_pre = cell_cycle_df_pre.join(final_output_df_barcodes[['oDEPool7_id','Barcode','sgRNA','Closest Hamming Distance', 'EcoWG1_id', 'Gene', 'N Mismatch', 'Category', 'TargetID', 'barcodeid']],how='inner')
n_obs = cell_cycle_df_ss.groupby("oDEPool7_id",sort=False).apply(lambda x: len(x.index.unique()), meta=int).compute()
n_obs = pd.DataFrame(n_obs).rename({0:"N Observations"}, axis=1).sort_index()
cell_cycle_df_ss = cell_cycle_df_ss.merge(n_obs,on="oDEPool7_id",how='inner')
cell_cycle_df_pre = cell_cycle_df_pre.merge(n_obs,on="oDEPool7_id",how='inner')
cell_cycle_df_ss.to_parquet("/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/2023-04-24_lDE28_Lineage_Cell_Cycle_Timeseries_Steady_State/", engine="pyarrow", overwrite=True)
cell_cycle_df_pre.to_parquet("/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/2023-04-24_lDE28_Lineage_Cell_Cycle_Timeseries_Preinduction/", engine="pyarrow", overwrite=True)
print("Cell Cycle Done.")

timepoints_df_ss = timepoints_df_ss.join(final_output_df_barcodes[['oDEPool7_id','Barcode','sgRNA','Closest Hamming Distance', 'EcoWG1_id', 'Gene', 'N Mismatch', 'Category', 'TargetID', 'barcodeid']],how='inner')
timepoints_df_pre = timepoints_df_pre.join(final_output_df_barcodes[['oDEPool7_id','Barcode','sgRNA','Closest Hamming Distance', 'EcoWG1_id', 'Gene', 'N Mismatch', 'Category', 'TargetID', 'barcodeid']],how='inner')
n_obs = timepoints_df_ss.groupby("oDEPool7_id",sort=False).apply(lambda x: len(x.index.unique()), meta=int).compute()
n_obs = pd.DataFrame(n_obs).rename({0:"N Observations"}, axis=1).sort_index()
timepoints_df_ss = timepoints_df_ss.merge(n_obs,on="oDEPool7_id",how='inner')
timepoints_df_pre = timepoints_df_pre.merge(n_obs,on="oDEPool7_id",how='inner')
timepoints_df_ss.to_parquet("/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/2023-04-24_lDE28_Lineage_Timepoints_Timeseries_Steady_State/", engine="pyarrow", overwrite=True)
timepoints_df_pre.to_parquet("/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/2023-04-24_lDE28_Lineage_Timepoints_Timeseries_Preinduction/", engine="pyarrow", overwrite=True)
print("Timepoints Done.")

growth_df_ss = growth_df_ss.join(final_output_df_barcodes[['oDEPool7_id','Barcode','sgRNA','Closest Hamming Distance', 'EcoWG1_id', 'Gene', 'N Mismatch', 'Category', 'TargetID', 'barcodeid']],how='inner')
growth_df_pre = growth_df_pre.join(final_output_df_barcodes[['oDEPool7_id','Barcode','sgRNA','Closest Hamming Distance', 'EcoWG1_id', 'Gene', 'N Mismatch', 'Category', 'TargetID', 'barcodeid']],how='inner')
n_obs = growth_df_ss.groupby("oDEPool7_id",sort=False).apply(lambda x: len(x.index.unique()), meta=int).compute()
n_obs = pd.DataFrame(n_obs).rename({0:"N Observations"}, axis=1).sort_index()
growth_df_ss = growth_df_ss.merge(n_obs,on="oDEPool7_id",how='inner')
growth_df_pre = growth_df_pre.merge(n_obs,on="oDEPool7_id",how='inner')
growth_df_ss.to_parquet("/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/2023-04-24_lDE28_Lineage_Growth_Timeseries_Steady_State/", engine="pyarrow", overwrite=True)
growth_df_pre.to_parquet("/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/2023-04-24_lDE28_Lineage_Growth_Timeseries_Preinduction/", engine="pyarrow", overwrite=True)
print("Growth Done.")
dask_controller.reset_worker_memory()

### Calculate Means, Variances and CVs of Measured Quantities

In [ ]:
def get_mean_var_and_cv(df,final_columns,columns_to_apply_aggregation,sgrna_key='oDEPool7_id'):
    sgrna_sorted = df.reset_index(drop=True).set_index(sgrna_key).persist()
    sgrna_groupby = sgrna_sorted.groupby(sgrna_key)
    mean_df = sgrna_groupby[columns_to_apply_aggregation].aggregate(np.mean).persist()
    std_df = sgrna_groupby[columns_to_apply_aggregation].aggregate(np.std).persist()
    cv_df = std_df/mean_df
    var_df = sgrna_groupby[columns_to_apply_aggregation].aggregate(np.var).persist()
    
    mean_df = mean_df.rename(columns={key: "Mean: " + key for key in mean_df.columns.tolist()})
    var_df = var_df.rename(columns={key: "Variance: " + key for key in var_df.columns.tolist()})
    cv_df = cv_df.rename(columns={key: "CV: " + key for key in cv_df.columns.tolist()})
    
    output_df = sgrna_groupby.first()
    output_df = output_df[final_columns]
    output_df = output_df.join(mean_df)
    output_df = output_df.join(var_df)
    output_df = output_df.join(cv_df)
    
    return output_df

In [ ]:
cell_cycle_df_ss = dd.read_parquet("/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/2023-04-24_lDE28_Lineage_Cell_Cycle_Timeseries_Steady_State",engine="pyarrow",calculate_divisions=True)
cell_cycle_df_pre = dd.read_parquet("/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/2023-04-24_lDE28_Lineage_Cell_Cycle_Timeseries_Preinduction",engine="pyarrow",calculate_divisions=True)
# 2023-03-25_lDE28_Run_1/2023-04-24_lDE28
timepoints_df_ss = dd.read_parquet("/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/2023-04-24_lDE28_Lineage_Timepoints_Timeseries_Steady_State",engine="pyarrow",calculate_divisions=True)
timepoints_df_pre = dd.read_parquet("/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/2023-04-24_lDE28_Lineage_Timepoints_Timeseries_Preinduction",engine="pyarrow",calculate_divisions=True)

growth_df_ss = dd.read_parquet("/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/2023-04-24_lDE28_Lineage_Growth_Timeseries_Steady_State",engine="pyarrow",calculate_divisions=True)
growth_df_pre = dd.read_parquet("/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/2023-04-24_lDE28_Lineage_Growth_Timeseries_Preinduction",engine="pyarrow",calculate_divisions=True)

final_columns = ['sgRNA','EcoWG1_id', 'Gene', 'N Mismatch', 'Category', 'TargetID', 'N Observations']

cell_cycle_df_ss_mean_cv = get_mean_var_and_cv(cell_cycle_df_ss,final_columns,cell_cycle_params,sgrna_key='oDEPool7_id').compute()
timepoints_df_ss_mean_cv = get_mean_var_and_cv(timepoints_df_ss,final_columns,timepoints_params,sgrna_key='oDEPool7_id').compute()
growth_df_ss_mean_cv = get_mean_var_and_cv(growth_df_ss,final_columns,growth_params,sgrna_key='oDEPool7_id').compute()

timepoints_df_ss_mean_cv = timepoints_df_ss_mean_cv[["Mean: " + item for item in timepoints_params] + ["Variance: " + item for item in timepoints_params] + ["CV: " + item for item in timepoints_params]]
growth_df_ss_mean_cv = growth_df_ss_mean_cv[["Mean: " + item for item in growth_params] + ["Variance: " + item for item in growth_params] + ["CV: " + item for item in growth_params]]

output_ss_meancv = pd.concat([cell_cycle_df_ss_mean_cv,timepoints_df_ss_mean_cv,growth_df_ss_mean_cv],axis=1).sort_index()
output_ss_meancv.to_pickle("/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/2023-04-24_lDE28_Lineage_Cell_Cycle_Timeseries_Steady_State_MeanCV.pkl")

del cell_cycle_df_ss_mean_cv
del timepoints_df_ss_mean_cv
del growth_df_ss_mean_cv
del output_ss_meancv

cell_cycle_df_pre_mean_cv = get_mean_var_and_cv(cell_cycle_df_pre,final_columns,cell_cycle_params,sgrna_key='oDEPool7_id').compute()
timepoints_df_pre_mean_cv = get_mean_var_and_cv(timepoints_df_pre,final_columns,timepoints_params,sgrna_key='oDEPool7_id').compute()
growth_df_pre_mean_cv = get_mean_var_and_cv(growth_df_pre,final_columns,growth_params,sgrna_key='oDEPool7_id').compute()

timepoints_df_pre_mean_cv = timepoints_df_pre_mean_cv[["Mean: " + item for item in timepoints_params] + ["Variance: " + item for item in timepoints_params] + ["CV: " + item for item in timepoints_params]]
growth_df_pre_mean_cv = growth_df_pre_mean_cv[["Mean: " + item for item in growth_params] + ["Variance: " + item for item in growth_params] + ["CV: " + item for item in growth_params]]

output_pre_meancv = pd.concat([cell_cycle_df_pre_mean_cv,timepoints_df_pre_mean_cv,growth_df_pre_mean_cv],axis=1).sort_index()
output_pre_meancv.to_pickle("/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/2023-04-24_lDE28_Lineage_Cell_Cycle_Timeseries_Preinduction_MeanCV.pkl")

In [ ]:
dask_controller.shutdown()